# PriorModel — Colab setup

This notebook installs **Julia 1.12.4**, clones the repo, mounts Drive, instantiates the project, and fetches or builds training data.

**Before anything else:** `Runtime → Change runtime type → Hardware accelerator → GPU` (T4 is enough).

Keep the kernel as **Python 3**. Julia is invoked with `!julia --project=/content/PriorModel ...` so we never depend on Colab's built-in Julia 1.10 runtime.

## Do not use `--project=.`

Colab's working directory is `/content`. Running `julia --project=.` from there creates a leftover **`/content/Project.toml`** that hides this repository's environment. After that, `using HDF5` and `using LuxCUDA` fail with "package not found in current path".

Always pass the clone path:

```bash
julia --project=/content/PriorModel …
```

If `/content/Project.toml` already exists, delete it in the next cell.

In [ ]:
%%bash
set -euo pipefail
if [[ -f /content/Project.toml ]]; then
  echo "Removing leftover /content/Project.toml (created by --project=.)"
  rm -f /content/Project.toml /content/Manifest.toml
fi
ls -la /content/Project.toml 2>/dev/null || echo "No /content/Project.toml — good."

## Install Julia 1.12.4

In [ ]:
%%bash
set -euo pipefail
JULIA_VERSION="1.12.4"
JULIA_VER="${JULIA_VERSION%.*}"
if julia --version 2>/dev/null | grep -q "${JULIA_VERSION}"; then
  julia --version
  exit 0
fi
echo "Installing Julia ${JULIA_VERSION}…"
URL="https://julialang-s3.julialang.org/bin/linux/x64/${JULIA_VER}/julia-${JULIA_VERSION}-linux-x86_64.tar.gz"
wget -q "${URL}" -O /tmp/julia.tar.gz
tar -xzf /tmp/julia.tar.gz -C /usr/local --strip-components=1
rm /tmp/julia.tar.gz
julia --version

## Clone the repository

In [ ]:
%%bash
set -euo pipefail
if [[ -d /content/PriorModel/.git ]]; then
  git -C /content/PriorModel pull --ff-only || true
else
  git clone https://github.com/hayrunnisayildiz/PriorModel.git /content/PriorModel
fi
ls /content/PriorModel/Project.toml

## Mount Google Drive

Keep large artifacts under `/content/drive/MyDrive/PriorModel/` with the same layout as the repo:

- `data/synthetic/train_pairs.h5` (and optional `train_pairs_v7.h5`)
- `models/*.jld2` checkpoints

Those paths are gitignored on purpose (see `.gitignore`).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Instantiate the Julia project

`MTGeophysics.jl` is pulled from GitHub tag **v0.4.2** (`Project.toml` `[sources]`), not from a local path.

**GLMakie warning:** MTGeophysics lists GLMakie as a hard dependency and imports it at package load. Instantiate still downloads it. On this headless VM, `src/pkg_setup.jl` skips auto-precompile so instantiate does not open an OpenGL context. Do **not** `using MTGeophysics` here; training must use `--commemi-every 0` (see last cell).

In [ ]:
%%bash
set -euo pipefail
export JULIA_PKG_PRECOMPILE_AUTO=0
export GKSwstype=nul
julia --project=/content/PriorModel -e 'using Pkg; Pkg.instantiate(); println("active=", Base.active_project())'

## Fetch data from Drive or build a small synthetic set

`scripts/colab_fetch_or_build.jl` copies `data/synthetic/*.h5` and `models/*.jld2` from Drive when present; otherwise it runs `build_train_pairs.jl` with `--n` (default 50).

In [ ]:
%%bash
set -euo pipefail
export JULIA_PKG_PRECOMPILE_AUTO=0
export GKSwstype=nul
julia --project=/content/PriorModel \
  /content/PriorModel/scripts/colab_fetch_or_build.jl \
  --drive-root /content/drive/MyDrive/PriorModel \
  --n 50

## Train

On Colab you **must** pass `--commemi-every 0` and `--no-plot`.

MTGeophysics.jl cannot be used without loading GLMakie (hard `using GLMakie` in the package module). The COMMEMI short-VFSA probe therefore cannot run on this headless VM unless you wrap Julia in `xvfb-run`, which is not the default here. `--no-plot` skips the training-curve PNG (Plots/GR).

The `--project=/content/PriorModel` flag is required; do not replace it with `--project=.`.

In [ ]:
%%bash
set -euo pipefail
export JULIA_PKG_PRECOMPILE_AUTO=0
export GKSwstype=nul
julia --project=/content/PriorModel \
  /content/PriorModel/src/training/train_mt_resistivity.jl \
  --dataset /content/PriorModel/data/synthetic/train_pairs.h5 \
  --commemi-every 0 --no-plot